In [1]:
import pandas as pd
import duckdb

In [2]:
con = duckdb.connect()

In [3]:
hum_degree_file = "./data/human_protein_degrees.csv"
hum_degree = pd.read_csv(hum_degree_file)
hum_degree = hum_degree.sort_values("degree_human", ascending=False).reset_index(drop=True).head(10)

hum_degree

,protein,degree_human
0,F7VJQ1,901198
1,Q9Y252,900024
2,O00165,899894
3,Q9H0V1,899554
4,Q9UHA4,899336
5,Q9UNL2,899020
6,P62993,898854
7,Q8WWA0,898807
8,Q9NVW2,898272
9,Q9NPJ8,897977


In [4]:
hum_prot_df = pd.DataFrame({"prot": hum_degree["protein"]})
con.register("prot_table", hum_prot_df)

hum_query = """
    SELECT p.*
    FROM parquet_scan('./data/inference_results/per_protein_results/*.parquet') p
    INNER JOIN prot_table t
    ON p.human_protein = t.prot
    WHERE p.score = 1
    """

top_human_interactions = con.execute(hum_query).df()
top_human_interactions

,human_protein,bacterial_protein,score
0,O00165,R5N1R7,1.0
1,O00165,A0A173W6J8,1.0
2,O00165,Q8R602,1.0
3,O00165,C7HTX5,1.0
4,O00165,R6V9Z1,1.0
...,...,...,...
6545687,O00165,A0A6M0L7E4,1.0
6545688,O00165,R9HSW2,1.0
6545689,O00165,A0A6I3RXL6,1.0
6545690,O00165,A0A173T8G7,1.0


In [5]:
bac_degree_file = "./data/bacterial_protein_degrees.csv"
bac_degree = pd.read_csv(bac_degree_file)
bac_degree = bac_degree.sort_values("degree_bacterial", ascending=False).reset_index(drop=True).head(10)

bac_degree

,protein,degree_bacterial
0,N9V9L9,9782
1,K0WYI1,9782
2,A0A1H4H2Y1,9782
3,Q8RFD9,9782
4,A0A1H3ZQT9,9782
5,A0A1K1LL05,9782
6,S0IPY0,9782
7,A0A1Y3X3C3,9782
8,F4XF02,9782
9,C9KJF4,9782


In [6]:
bac_prot_df = pd.DataFrame({"prot": bac_degree["protein"]})
con.register("prot_table", bac_prot_df)

bac_query = """
    SELECT p.*
    FROM parquet_scan('./data/inference_results/per_protein_results/*.parquet') p
    INNER JOIN prot_table t
    ON p.bacterial_protein = t.prot
    WHERE p.score = 1
    """

top_bac_interactions = con.execute(bac_query).df()
top_bac_interactions

,human_protein,bacterial_protein,score
0,A0A087X179,F4XF02,1.0
1,A0A087X179,A0A1K1LL05,1.0
2,A0A087X179,A0A1H3ZQT9,1.0
3,A0A087X179,Q8RFD9,1.0
4,A1L0T0,K0WYI1,1.0
...,...,...,...
96171,Q9Y6X8,C9KJF4,1.0
96172,Q9Y6X8,A0A1H3ZQT9,1.0
96173,Q9Y6X8,N9V9L9,1.0
96174,Q9Y6X8,A0A1Y3X3C3,1.0


In [ ]:
# Top human interactions are larger than top bacterial interactions. Lets do a random sample.
top_human_interactions = top_human_interactions.sample(n=top_bac_interactions.shape[0], random_state=42)
# Clean the dataframe
top_human_interactions = top_human_interactions[["human_protein", "bacterial_protein"]].reset_index(drop=True)
top_human_interactions.to_csv("./data/suppl_sample_interactions_top_human.csv", index=False)
top_human_interactions


,human_protein,bacterial_protein
0,Q9Y252,A0A3D8HAJ6
1,F7VJQ1,A0A3E5BCD6
2,Q9UNL2,R9HX35
3,Q9NVW2,A0A3D8HD11
4,Q9NVW2,A0A845R1G0
...,...,...
96171,Q9H0V1,A0A415CEH0
96172,Q9UNL2,A0A0M6X0V4
96173,P62993,A0A851HJM7
96174,Q9NVW2,A0A4P6LRM3


In [ ]:
# Clean the bac dataframe as well.
top_bac_interactions = top_bac_interactions[["human_protein", "bacterial_protein"]].reset_index(drop=True)
top_bac_interactions.to_csv("./data/suppl_sample_interactions_top_bac.csv", index=False)
top_bac_interactions

,human_protein,bacterial_protein
0,A0A087X179,F4XF02
1,A0A087X179,A0A1K1LL05
2,A0A087X179,A0A1H3ZQT9
3,A0A087X179,Q8RFD9
4,A1L0T0,K0WYI1
...,...,...
96171,Q9Y6X8,C9KJF4
96172,Q9Y6X8,A0A1H3ZQT9
96173,Q9Y6X8,N9V9L9
96174,Q9Y6X8,A0A1Y3X3C3


In [7]:
# Read all sheets from S1

'''perfixis = [
    "metabol",
    "diab",
    "gluc",
    "glyc",
    "lipid",
    "hear",
    "cardio"
]

xls = pd.ExcelFile(S_path)
res_df_list = []
for sheet_name in xls.sheet_names:
    if "Human" in sheet_name:
        print(sheet_name)
        df = pd.read_excel(xls, sheet_name=sheet_name)
        
        # Keep rows that contain the perfix either on Gene Ontology (biological process) or Gene Ontology (molecular function) columns
        sub_dfs = []
        for prefix in perfixis:
            sub_df = df[df["Gene Ontology (biological process)"].str.contains(prefix) | df["Gene Ontology (molecular function)"].str.contains(prefix)]
            sub_dfs.append(sub_df)
        sub_df = pd.concat(sub_dfs, axis=0)

        print(len(sub_df))
        
        # Short by degree
        sub_df = sub_df.sort_values("Degree", ascending=False).reset_index(drop=True).head(10)

        
        
        # To list
        res_df_list.append(sub_df)

res_df = pd.concat(res_df_list, axis=0).reset_index(drop=True)
# Drop duplicates based on Protein
res_df = res_df.drop_duplicates(subset="Protein", keep="first")
res_df.to_csv("./data/paper_supl/S1.csv", index=False)    '''        

'perfixis = [\n    "metabol",\n    "diab",\n    "gluc",\n    "glyc",\n    "lipid",\n    "hear",\n    "cardio"\n]\n\nxls = pd.ExcelFile(S_path)\nres_df_list = []\nfor sheet_name in xls.sheet_names:\n    if "Human" in sheet_name:\n        print(sheet_name)\n        df = pd.read_excel(xls, sheet_name=sheet_name)\n\n        # Keep rows that contain the perfix either on Gene Ontology (biological process) or Gene Ontology (molecular function) columns\n        sub_dfs = []\n        for prefix in perfixis:\n            sub_df = df[df["Gene Ontology (biological process)"].str.contains(prefix) | df["Gene Ontology (molecular function)"].str.contains(prefix)]\n            sub_dfs.append(sub_df)\n        sub_df = pd.concat(sub_dfs, axis=0)\n\n        print(len(sub_df))\n\n        # Short by degree\n        sub_df = sub_df.sort_values("Degree", ascending=False).reset_index(drop=True).head(10)\n\n\n\n        # To list\n        res_df_list.append(sub_df)\n\nres_df = pd.concat(res_df_list, axis=0).rese